In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
TRAIN = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
TEST = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
SAMPLE = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [3]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


### ------------------------------- ROBERTA ----------------------------------

In [5]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice

model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(TRAIN, test_size=0.2, random_state=42, stratify=TRAIN["answer"])

In [7]:
from torch.utils.data import Dataset, DataLoader

class RobertaDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_length
        self.label_map = {
            "A": 0,
            "B": 1,
            "C": 2,
            "D": 3,
            "E": 4
        }
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        prompt = row["prompt"]
        choices=[row["A"], row["B"], row["C"], row["D"], row["E"]]
        label = self.label_map[row["answer"]]
        tokenized = tokenizer([prompt]*5, choices, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return {"input_ids": tokenized["input_ids"],
               "attention_mask": tokenized["attention_mask"],
               "labels": torch.tensor(label, dtype=torch.long)}

In [8]:
lengths = []

for _, row in TRAIN.iterrows():
    prompt = row["prompt"]
    choices = [row["A"], row["B"], row["C"], row["D"], row["E"]]

    enc = tokenizer(
        [prompt] * 5,
        choices,
        truncation=False,
        padding=False
    )

    lengths.extend(len(ids) for ids in enc["input_ids"])

print("Max token length:", max(lengths))
print("Average token length:", sum(lengths) / len(lengths))

Max token length: 200
Average token length: 61.4702


In [9]:
train_dataset = RobertaDataset(data=train_df, tokenizer=tokenizer, max_length=200)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [10]:
val_dataset = RobertaDataset(data=test_df, tokenizer=tokenizer, max_length=200)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [11]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

learning_rate = 2e-5
epochs = 5
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=25, num_training_steps=250)

In [12]:
wandb.init(
    project="Smart_MCQ_solver",
    name="roberta-ensemble-run1",
    config={
        "epochs": epochs,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "batch_size": train_dataloader.batch_size,
        "model": "roberta-base"
    }
)

In [13]:
model = model.to(device)
model.train()

torch.cuda.empty_cache()

global_step = 0

for epoch in range(epochs):
    total_loss = 0

    for step, batch in enumerate(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        wandb.log({
            "train/batch_loss": loss.item(),
            "learning_rate": scheduler.get_last_lr()[0],
            "global_step": global_step
        })
        global_step += 1

        del outputs, loss

    avg_loss = total_loss / len(train_dataloader)

    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")
    wandb.log({"train/epoch_loss": avg_loss, "epoch": epoch + 1})

Epoch 1: Loss = 1.5299
Epoch 2: Loss = 0.8013
Epoch 3: Loss = 0.4655
Epoch 4: Loss = 0.4483
Epoch 5: Loss = 0.4433


In [14]:
model.eval()

val_loss = 0
scores = []

with torch.no_grad():

    for batch in val_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits
        val_loss += loss.item()

        top3 = torch.topk(logits, k=3, dim=1).indices

        labels = batch["labels"]

        for true_label, pred in zip(labels, top3):

            pred = pred.tolist()
            true_label = true_label.item()

            if true_label in pred:
                ap = 1.0 / (pred.index(true_label) + 1)
            else:
                ap = 0.0

            scores.append(ap)

avg_val_loss = val_loss / len(val_dataloader)
val_map3 = sum(scores) / len(scores)

print(f"Validation Loss : {avg_val_loss:.4f}")
print(f"Validation MAP@3 : {val_map3:.4f}")

wandb.log({
    "val/loss": avg_val_loss,
    "val/MAP@3": val_map3
})
wandb.finish()

Validation Loss : 0.3187
Validation MAP@3 : 0.9646


epoch,▁▃▅▆█
global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
learning_rate,▃▄████▇▇▆▆▅▅▅▄▄▄▄▄▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/batch_loss,█████▇▅▄▄▄▄▄▃▄▂▂▃▂▂▄▂▃▁▂▂▃▃▃▂▂▂▃▂▃▃▂▂▃▂▂
train/epoch_loss,█▃▁▁▁
val/MAP@3,▁
val/loss,▁
epoch,5
global_step,499
learning_rate,0
train/batch_loss,0.52253


In [15]:
class RobertaTestDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        prompt = row["prompt"]
        choices = [row["A"], row["B"], row["C"], row["D"], row["E"]]

        tokenized = self.tokenizer(
            [prompt] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"]
        }

In [18]:
test_dataset = RobertaTestDataset(TEST, tokenizer, max_length=256)

test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [19]:
model.eval()
roberta_probs = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].squeeze(1).to(device)
        attention_mask = batch["attention_mask"].squeeze(1).to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        probs = torch.softmax(outputs.logits, dim=1)
        roberta_probs.append(probs.cpu())

roberta_probs = torch.cat(roberta_probs)

### ------------------------------- Bi-LSTM ----------------------------------

In [21]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
class BiLSTMDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_length

        self.label_map = {
            "A": 0,
            "B": 1,
            "C": 2,
            "D": 3,
            "E": 4
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        text = (
            f"Question: {row['prompt']} "
            f"A. {row['A']} "
            f"B. {row['B']} "
            f"C. {row['C']} "
            f"D. {row['D']} "
            f"E. {row['E']}"
        )

        encoding = self.tokenizer(text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        label = self.label_map[row["answer"]]

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [23]:
train_df, test_df = train_test_split(TRAIN, test_size=0.2, random_state=42, stratify=TRAIN["answer"])

In [24]:
train_dataset = BiLSTMDataset(data=train_df, tokenizer=tokenizer, max_length=256)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [31]:
val_dataset = BiLSTMDataset(data=test_df, tokenizer=tokenizer, max_length=256)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [26]:
import torch.nn as nn 

class BiLSTM(nn.Module):
    def __init__(self, vocab_size=len(train_dataset.tokenizer), embed_dim=128, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim*2, 5)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        output, (hidden, _) = self.lstm(x)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(hidden)

In [27]:
from torch.optim import AdamW

learning_rate = 1e-3
epochs = 20
model = BiLSTM()
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

In [28]:
wandb.init(
    project="Smart_MCQ_solver",
    name="bi-lstm-ensemble",
    config={
        "epochs": epochs,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "batch_size": train_dataloader.batch_size,
        "model": "bi-LSTM"
    }
)

In [29]:
model = model.to(device)
model.train()

global_step = 0
for epoch in range(epochs):
    total_loss = 0

    for batch in train_dataloader:
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        output = model(input_ids)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        wandb.log({
            "train/batch_loss": loss.item(),
            "learning_rate": learning_rate,
            "global_step": global_step
        })
        global_step += 1
    avg_loss = total_loss / len(train_dataloader)    
    print(f"epoch : {epoch + 1}, Loss : {total_loss: .4f}")
    wandb.log({"train/epoch_loss": avg_loss, "epoch": epoch + 1})

epoch : 1, Loss :  80.4047
epoch : 2, Loss :  73.5390
epoch : 3, Loss :  60.5603
epoch : 4, Loss :  51.3020
epoch : 5, Loss :  42.0716
epoch : 6, Loss :  36.4959
epoch : 7, Loss :  24.1401
epoch : 8, Loss :  13.1487
epoch : 9, Loss :  12.5584
epoch : 10, Loss :  8.8795
epoch : 11, Loss :  4.1607
epoch : 12, Loss :  2.6733
epoch : 13, Loss :  1.5903
epoch : 14, Loss :  1.0073
epoch : 15, Loss :  0.6416
epoch : 16, Loss :  0.3245
epoch : 17, Loss :  0.1725
epoch : 18, Loss :  0.1152
epoch : 19, Loss :  0.0876
epoch : 20, Loss :  0.0719


In [32]:
model.eval()

correct = 0
total = 0
map3 = 0.0
val_loss = 0

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        top3 = outputs.topk(3, dim=1).indices

        for i in range(labels.size(0)):
            true_label = labels[i].item()
            predictions = top3[i].tolist()

            if true_label == predictions[0]:
                map3 += 1.0
            elif true_label == predictions[1]:
                map3 += 0.5
            elif true_label == predictions[2]:
                map3 += 1.0 / 3.0

accuracy = correct / total
map3 /= total


print(f"Validation Accuracy : {accuracy:.4f}")
print(f"Validation MAP@3    : {map3:.4f}")
wandb.log({
    "val/accuracy": accuracy,
    "val/MAP@3": map3
})
wandb.finish()

Validation Accuracy : 0.9950
Validation MAP@3    : 0.9975


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇████
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/batch_loss,███▆▆▆▅▆▆▄▃▃▃▄▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch_loss,█▇▆▅▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁
val/MAP@3,▁
val/accuracy,▁
epoch,20
global_step,999
learning_rate,0.001
train/batch_loss,0.0007


In [33]:
from torch.utils.data import Dataset

class BiLSTMTestDataset(Dataset):

    def __init__(self, data, tokenizer, max_length):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        text = (
            f"Question: {row['prompt']} "
            f"A. {row['A']} "
            f"B. {row['B']} "
            f"C. {row['C']} "
            f"D. {row['D']} "
            f"E. {row['E']}"
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

In [35]:
test_dataset = BiLSTMTestDataset(TEST, tokenizer, max_length=256)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [36]:
model.eval()

bilstm_probs = []

with torch.no_grad():

    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        logits = model(input_ids)
        probs = torch.softmax(logits, dim=1)
        bilstm_probs.append(probs.cpu())
bilstm_probs = torch.cat(bilstm_probs)

### ------------------------------- Ensemble ----------------------------------

In [40]:
final_probs = 0.9*bilstm_probs + 0.1*roberta_probs

In [41]:
label_map = ["A","B","C","D","E"]

top3 = torch.topk(final_probs,3,dim=1).indices.numpy()

predictions = []

for pred in top3:
    predictions.append(" ".join(label_map[i] for i in pred))

In [42]:
submission = pd.DataFrame({
    "id": TEST["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)
submission.head()

,id,prediction
0,1,A D B
1,2,B A E
2,3,B A E
3,4,E C A
4,5,C A E
